In [ ]:
# 1. load the tokenizer trained by bpe_tokenization.py
import sys
from pathlib import Path

HERE = Path.cwd()
if not (HERE / "bpe_tokenization.py").exists():        # notebook launched from the repo root
    HERE = HERE / "experiments" / "tokenization"
sys.path.insert(0, str(HERE))

from bpe_tokenization import BPETokenizer, render

tok = BPETokenizer.from_pretrained()

print(f"vocab size     : {tok.n_vocab:,}")
print(f"learned merges : {len(tok.merges):,}")
print(f"special tokens : {', '.join(tok.special_tokens)}")
print(f"<|endoftext|>  : {tok.eot_id}     <|pad|> : {tok.pad_id}")
print(f"trained on     : {tok.meta.get('corpus_bytes', 0) / 1e9:.2f} GB")

In [ ]:
# 2. encode -> decode on natural English (nothing sampled from the stories)
TEXTS = [
    "The quick brown fox jumps over the lazy dog.",
    "She said, \"I can't believe it's already 5 o'clock!\"",
    "In 2024, researchers published 1,247 papers on transformer architectures.",
    "Backpropagation computes gradients by applying the chain rule in reverse.",
    "Email me at hello@example.com -- or don't; either way, it's fine.",
    "Café, naïve, jalapeño and 日本語 survive byte-for-byte. \U0001f389",
    "Line one.\nLine two follows a newline,\tand a tab.",
    "   leading and trailing whitespace   ",
]

print(f"{'ok':<6}{'tokens':>7}{'bytes/tok':>11}   text")
print("-" * 96)

all_ok = True
for text in TEXTS:
    ids = tok.encode(text)
    back = tok.decode(ids)
    ok = back == text
    all_ok = all_ok and ok
    print(f"{'PASS' if ok else 'FAIL':<6}{len(ids):>7}{len(text.encode()) / len(ids):>11.2f}   {text!r}")

print("-" * 96)
print("all strings decoded back to the exact original" if all_ok else "MISMATCH FOUND")

In [ ]:
# 3. your own text: encode -> ids -> decode -> compare
DEFAULT = "Tokenizers turn text into integers, and integers back into text."

try:
    TEXT = input("Text to tokenize (press Enter for the default): ").strip()
except (EOFError, OSError):
    TEXT = ""
TEXT = TEXT or DEFAULT

ids = tok.encode(TEXT)
back = tok.decode(ids)

print(f"\ninput   : {TEXT!r}")
print(f"bytes   : {len(TEXT.encode())}")
print(f"tokens  : {len(ids)}   ({len(TEXT.encode()) / len(ids):.2f} bytes/token)")
print(f"ids     : {ids}")

print("\npiece by piece   (Ġ = space, Ċ = newline):")
for i in ids[:40]:
    print(f"  {i:>6}  {render(tok.vocab[i])}")
if len(ids) > 40:
    print(f"  ... {len(ids) - 40} more")

print(f"\ndecoded : {back!r}")
print(f"\nidentical to input: {back == TEXT}")